<a href="https://colab.research.google.com/github/iaportnov/GP_2_real_estate_investment_strategy/blob/main/feature_construction_filters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import requests
import pandas as pd
import numpy as np
import json

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

In [70]:
!wget https://raw.githubusercontent.com/iaportnov/GP_2_real_estate_investment_strategy/refs/heads/main/area_sales_community_df.csv

--2026-04-09 18:07:08--  https://raw.githubusercontent.com/iaportnov/GP_2_real_estate_investment_strategy/refs/heads/main/area_sales_community_df.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2915727 (2.8M) [text/plain]
Saving to: ‘area_sales_community_df.csv’

area_sales_communit 100%[===================>]   2.78M  --.-KB/s    in 0.03s   

2026-04-09 18:07:08 (108 MB/s) - ‘area_sales_community_df.csv’ saved [2915727/2915727]



In [72]:
df = pd.read_csv('area_sales_community_df.csv', index_col=0)
df

,latitude,longitude,area,geo_id,homeSaleCount_2020,avgSalePrice_2020,medSalePrice_2020,homeSaleCount_2021,avgSalePrice_2021,medSalePrice_2021,...,housing_Built_1970_1979_Pct,housing_Built_1960_1969_Pct,housing_Built_1950_1959_Pct,housing_Built_1940_1949_Pct,housing_Built_1939_Or_Earlier_Pct,travel_Time_To_Work_0_14_Mi_Pct,travel_Time_To_Work_15_29_Mi_Pct,travel_Time_To_Work_30_59_Mi_Pct,travel_Time_To_Work_60_89_Mi_Pct,travel_Time_To_Work_90_Or_More_Mi_Pct
0,27.039971,-82.074833,1.327111,9cd7eab982c5c35bcd6f05d732d333c5,10.0,84667.0,29900.0,24.0,85341.0,29900.0,...,0.80,0.77,0.79,0.67,0.67,17.40,53.64,22.81,6.14,0.01
1,26.280449,-80.253057,0.285343,b508f5dc4143a48154f0ff995b081678,7.0,229071.0,117000.0,12.0,609375.0,739750.0,...,25.51,10.77,0.00,0.00,0.00,25.10,46.68,28.22,0.01,0.00
2,34.634010,-86.575343,0.524357,5b55d57472de1efb82e58cc003f53b78,51.0,326056.0,333005.0,24.0,238392.0,267500.0,...,9.62,10.42,2.59,0.40,0.38,23.30,55.78,19.09,0.69,1.15
3,39.827121,-84.173380,4.485721,6a62dc1d0f1a59ae0c3de95037e1b683,2.0,47500.0,47500.0,0.0,0.0,0.0,...,23.44,3.60,9.26,5.73,7.48,39.76,43.39,15.80,1.06,0.00
4,34.195130,-84.295753,1.031886,d7926013088462e0208e1edce50bf340,16.0,463266.0,156885.0,30.0,674595.0,182500.0,...,5.52,6.17,0.47,0.36,0.36,1.56,42.18,50.25,0.07,5.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4515,26.145432,-80.213790,0.562633,d1e3095ab678803f4b041f4235798872,69.0,260311.0,280000.0,52.0,307205.0,317500.0,...,40.44,21.47,2.00,2.11,0.78,10.55,46.26,38.15,3.30,1.74
4516,28.044742,-81.936377,0.355987,50ebe9038c385b6117d18e557d076841,32.0,158539.0,160500.0,36.0,178502.0,176000.0,...,19.33,8.05,14.85,2.96,7.52,15.17,44.05,37.75,1.99,1.05
4517,44.954249,-93.364255,0.375201,3c969346ea2aa69d1b17362ecb262c76,67.0,315080.0,333000.0,100.0,338246.0,335500.0,...,0.83,5.68,27.43,29.19,8.30,20.02,47.77,19.83,3.78,8.60
4518,40.641697,-74.109807,0.183835,a6498e25964d71c4c3834d3f9c4bb11a,14.0,511885.0,421250.0,30.0,549579.0,545000.0,...,2.95,8.42,7.39,10.31,31.37,16.64,24.05,27.31,22.73,9.28


Функция для сплита колонки geographyName:

In [73]:
def get_geography_names(geography_name):
  all_names = geography_name.split(',')
  if len(all_names) == 3:
    return [None, all_names[0], all_names[1], all_names[2]]
  return all_names

In [74]:
splited = df['geographyName'].str.split(', ', expand=True)
df[['neighborhood', 'city', 'county', 'state']] = df['geographyName'].apply(lambda x: pd.Series(get_geography_names(x)))
df = df.drop('geographyName', axis=1)

Переименуем кое-какие колонки:

In [75]:
df.rename(columns={
    'healthcare': 'healthcare_count',
    'grocery': 'grocery_count',
    'education': 'education_count',
    'entertainment': 'entertainment_count',
    'recreation': 'recreation_count'
}, inplace=True)

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4520 entries, 0 to 4519
Data columns (total 96 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   latitude                                         4520 non-null   float64
 1   longitude                                        4520 non-null   float64
 2   area                                             4520 non-null   float64
 3   geo_id                                           4520 non-null   object 
 4   homeSaleCount_2020                               4520 non-null   float64
 5   avgSalePrice_2020                                4520 non-null   float64
 6   medSalePrice_2020                                4520 non-null   float64
 7   homeSaleCount_2021                               4520 non-null   float64
 8   avgSalePrice_2021                                4520 non-null   float64
 9   medSalePrice_2021                  

### Конструирование признаков

In [77]:
def purchase_availability(row):
  value = row["housing_Owner_Households_Median_Value"]
  income = row["median_Household_Income"]

  if pd.isna(value) or pd.isna(income) or income <= 0:
    return pd.NA

  return value / income

# показывает, во сколько годовых медианных доходов обходится жилье

In [78]:
df["purchase_availability"] = df.apply(purchase_availability, axis=1)

In [79]:
def rent_availability(row):
  rent = row["housing_Median_Rent"]
  income = row["median_Household_Income"]

  if pd.isna(rent) or pd.isna(income) or income <= 0:
    return pd.NA

  return 12 * rent / income

# показывает, какая доля годового медианного дохода уходит на медианную аренду. умножаем на 12 потому что тут помесячная стоимость съема квартиры

In [80]:
df["rent_availability"] = df.apply(rent_availability, axis=1)

In [81]:
def expenditures_Idx(row):
  annual = row["costIndex_Annual_Expenditures"]
  housing = row["costIndex_Housing"]
  healthcare = row["costIndex_Healthcare"]
  education = row["costIndex_Education"]

  if pd.isna(annual) or pd.isna(housing) or pd.isna(healthcare) or pd.isna(education):
    return pd.NA

  return (annual + housing + healthcare + education) / 4

# чем выше, тем локация дороже по расходам

In [82]:
df["expenditures_Idx"] = df.apply(expenditures_Idx, axis=1)
df = df.drop(columns=["costIndex_Annual_Expenditures", "costIndex_Housing", "costIndex_Healthcare", "costIndex_Education"])

In [83]:
def development_density_Pct(row):
  pct_2 = row["housing_Occupied_Structure_2_Units_Pct"]
  pct_3_4 = row["housing_Occupied_Structure_3_4_Units_Pct"]
  pct_5_9 = row["housing_Occupied_Structure_5_9_Units_Pct"]
  pct_10_19 = row["housing_Occupied_Structure_10_19_Units_Pct"]
  pct_20_49 = row["housing_Occupied_Structure_20_49_Units_Pct"]
  pct_50_more = row["housing_Occupied_Structure_50_Or_More_Units_Pct"]

  return (pct_2 * 2 + pct_3_4 * 4 + pct_5_9 * 7 + pct_10_19 * 15 + pct_20_49 * 35 + pct_50_more * 50) / 6

# чем выше, тем выше доля более плотной многоквартирной застройки, то есть "муравейники"

In [84]:
df["development_density_Pct"] = df.apply(development_density_Pct, axis=1)
df = df.drop(columns=["housing_Occupied_Structure_2_Units_Pct", "housing_Occupied_Structure_3_4_Units_Pct", "housing_Occupied_Structure_5_9_Units_Pct", "housing_Occupied_Structure_10_19_Units_Pct", "housing_Occupied_Structure_20_49_Units_Pct", "housing_Occupied_Structure_50_Or_More_Units_Pct"])

In [85]:
def average_built_year(row):
  built_year_mapping = {
      "housing_Built_2010_Or_Later_Pct": 2015.0,
      "housing_Built_2000_2009_Pct": 2004.5,
      "housing_Built_1990_1999_Pct": 1994.5,
      "housing_Built_1980_1989_Pct": 1984.5,
      "housing_Built_1970_1979_Pct": 1974.5,
      "housing_Built_1960_1969_Pct": 1964.5,
      "housing_Built_1950_1959_Pct": 1954.5,
      "housing_Built_1940_1949_Pct": 1944.5,
      "housing_Built_1939_Or_Earlier_Pct": 1935.0}

  weighted_sum = 0
  total_pct = 0

  for col, year in built_year_mapping.items():

    value = row[col]
    if pd.isna(value):
      return pd.NA
    weighted_sum += value * year
    total_pct += value
  if total_pct <= 0:
    return pd.NA

  return weighted_sum / total_pct

# показываем средневзвешенный год постройки зданий

In [86]:
df["average_built_year"] = df.apply(average_built_year, axis=1)
df = df.drop(columns=["housing_Built_2010_Or_Later_Pct", "housing_Built_2000_2009_Pct", "housing_Built_1990_1999_Pct", "housing_Built_1980_1989_Pct", "housing_Built_1970_1979_Pct", "housing_Built_1960_1969_Pct", "housing_Built_1950_1959_Pct", "housing_Built_1940_1949_Pct", "housing_Built_1939_Or_Earlier_Pct"])

In [87]:
def accessible_work_Pct(row):
  work_0_14 = row["travel_Time_To_Work_0_14_Mi_Pct"]
  work_15_29 = row["travel_Time_To_Work_15_29_Mi_Pct"]
  work_30_59 = row["travel_Time_To_Work_30_59_Mi_Pct"]
  work_60_89 = row["travel_Time_To_Work_60_89_Mi_Pct"]
  work_90_more = row["travel_Time_To_Work_90_Or_More_Mi_Pct"]

  return (work_0_14 * 7 + work_15_29 * 22.5 + work_30_59 * 45 + work_60_89 * 75 + work_90_more * 90) / 5

# чем выше тем сложнее и дольше добираться до работы

In [88]:
df["accessible_work_Pct"] = df.apply(accessible_work_Pct, axis=1)
df = df.drop(columns=["travel_Time_To_Work_0_14_Mi_Pct", "travel_Time_To_Work_15_29_Mi_Pct", "travel_Time_To_Work_30_59_Mi_Pct", "travel_Time_To_Work_60_89_Mi_Pct", "travel_Time_To_Work_90_Or_More_Mi_Pct"])

In [89]:
df

,latitude,longitude,area,geo_id,homeSaleCount_2020,avgSalePrice_2020,medSalePrice_2020,homeSaleCount_2021,avgSalePrice_2021,medSalePrice_2021,...,neighborhood,city,county,state,purchase_availability,rent_availability,expenditures_Idx,development_density_Pct,average_built_year,accessible_work_Pct
0,27.039971,-82.074833,1.327111,9cd7eab982c5c35bcd6f05d732d333c5,10.0,84667.0,29900.0,24.0,85341.0,29900.0,...,Price End,North Port,Sarasota County,FL,3.645503,0.39358,99.00,23.025000,2004.459651,563.310
1,26.280449,-80.253057,0.285343,b508f5dc4143a48154f0ff995b081678,7.0,229071.0,117000.0,12.0,609375.0,739750.0,...,The Hills,Coral Springs,Broward County,FL,3.223138,0.098195,130.25,101.593333,1985.122807,499.330
2,34.634010,-86.575343,0.524357,5b55d57472de1efb82e58cc003f53b78,51.0,326056.0,333005.0,24.0,238392.0,267500.0,...,Whitesburg Estates,Huntsville,Madison County,AL,2.774031,0.11919,131.00,13.288333,1988.236361,486.490
3,39.827121,-84.173380,4.485721,6a62dc1d0f1a59ae0c3de95037e1b683,2.0,47500.0,47500.0,0.0,0.0,0.0,...,Kittyhawk,Dayton,Montgomery County,OH,1.948550,0.163636,103.75,7.216667,1977.952627,409.019
4,34.195130,-84.295753,1.031886,d7926013088462e0208e1edce50bf340,16.0,463266.0,156885.0,30.0,674595.0,182500.0,...,None,Echelon,Cherokee County,GA,3.474887,0.018414,142.75,0.000000,1997.400393,752.214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4515,26.145432,-80.213790,0.562633,d1e3095ab678803f4b041f4235798872,69.0,260311.0,280000.0,52.0,307205.0,317500.0,...,Flair,Lauderhill,Broward County,FL,4.479686,0.341566,56.25,173.603333,1978.593031,647.110
4516,28.044742,-81.936377,0.355987,50ebe9038c385b6117d18e557d076841,32.0,158539.0,160500.0,36.0,178502.0,176000.0,...,Lake Bonny,Lakeland,Polk County,FL,2.482318,0.230923,90.25,44.833333,1978.508727,607.963
4517,44.954249,-93.364255,0.375201,3c969346ea2aa69d1b17362ecb262c76,67.0,315080.0,333000.0,100.0,338246.0,335500.0,...,Bronx Park,St. Louis Park,Hennepin County,MN,3.021860,0.215081,133.50,0.000000,1965.122615,632.963
4518,40.641697,-74.109807,0.183835,a6498e25964d71c4c3834d3f9c4bb11a,14.0,511885.0,421250.0,30.0,549579.0,545000.0,...,Livingston,New York,Richmond County,NY,4.589924,0.151352,111.50,30.741667,1964.303948,885.301


### Фильтры

In [90]:
def filter_crime_index(df, t=100):
  return df['crime_Index'] <= t
# фильтруем по 100 тк это средний крайм по сша (потом мб подумать побольше поставить)

def population_Chg_filter(df, min_chg=1):
  return df['population_Chg_Pct_5_Yr_Projection'] >= min_chg
# посмотрели с сеней популяция почти не меняется

def median_Household_Income_filter(df, min_income=55000):
  return df['median_Household_Income'] >= min_income
# чтоб не совсем броуки гетто районы

def air_pollution_filter(df, max_pollution=70):
  return df['air_Pollution_Index'] <= max_pollution
# загрязнение воздуха

def unemployment_filter(df, max_unemployment=10):
  male_unemp = df['population_16P_Unemployed_Male_Pct']
  female_unemp = df['population_16P_Unemployed_Female_Pct']
  return (male_unemp + female_unemp) / 2 <= max_unemployment
# безработица тоже гетто вайб

def population_filter(df, min_population=30000):
  return df['population'] >= min_population
# тут можно оставить чисто популяцию, либо новый признак ввести
# df['population'] / df['area'] # person_per_sq_mile

In [91]:
df.shape

(4520, 78)

In [92]:
df_test = df.copy()

In [93]:
df_test[df_test.apply(filter_crime_index, axis=1)]

,latitude,longitude,area,geo_id,homeSaleCount_2020,avgSalePrice_2020,medSalePrice_2020,homeSaleCount_2021,avgSalePrice_2021,medSalePrice_2021,...,neighborhood,city,county,state,purchase_availability,rent_availability,expenditures_Idx,development_density_Pct,average_built_year,accessible_work_Pct
0,27.039971,-82.074833,1.327111,9cd7eab982c5c35bcd6f05d732d333c5,10.0,84667.0,29900.0,24.0,85341.0,29900.0,...,Price End,North Port,Sarasota County,FL,3.645503,0.39358,99.00,23.025000,2004.459651,563.310
1,26.280449,-80.253057,0.285343,b508f5dc4143a48154f0ff995b081678,7.0,229071.0,117000.0,12.0,609375.0,739750.0,...,The Hills,Coral Springs,Broward County,FL,3.223138,0.098195,130.25,101.593333,1985.122807,499.330
2,34.634010,-86.575343,0.524357,5b55d57472de1efb82e58cc003f53b78,51.0,326056.0,333005.0,24.0,238392.0,267500.0,...,Whitesburg Estates,Huntsville,Madison County,AL,2.774031,0.11919,131.00,13.288333,1988.236361,486.490
3,39.827121,-84.173380,4.485721,6a62dc1d0f1a59ae0c3de95037e1b683,2.0,47500.0,47500.0,0.0,0.0,0.0,...,Kittyhawk,Dayton,Montgomery County,OH,1.948550,0.163636,103.75,7.216667,1977.952627,409.019
4,34.195130,-84.295753,1.031886,d7926013088462e0208e1edce50bf340,16.0,463266.0,156885.0,30.0,674595.0,182500.0,...,None,Echelon,Cherokee County,GA,3.474887,0.018414,142.75,0.000000,1997.400393,752.214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4514,26.126417,-80.289843,0.241537,9337a11b79baeff93bc6563e2144d72e,31.0,476758.0,460000.0,23.0,534634.0,482500.0,...,Nob Hill Estates,Plantation,Broward County,FL,4.698648,0.251687,112.50,87.645000,1991.913948,558.462
4515,26.145432,-80.213790,0.562633,d1e3095ab678803f4b041f4235798872,69.0,260311.0,280000.0,52.0,307205.0,317500.0,...,Flair,Lauderhill,Broward County,FL,4.479686,0.341566,56.25,173.603333,1978.593031,647.110
4517,44.954249,-93.364255,0.375201,3c969346ea2aa69d1b17362ecb262c76,67.0,315080.0,333000.0,100.0,338246.0,335500.0,...,Bronx Park,St. Louis Park,Hennepin County,MN,3.021860,0.215081,133.50,0.000000,1965.122615,632.963
4518,40.641697,-74.109807,0.183835,a6498e25964d71c4c3834d3f9c4bb11a,14.0,511885.0,421250.0,30.0,549579.0,545000.0,...,Livingston,New York,Richmond County,NY,4.589924,0.151352,111.50,30.741667,1964.303948,885.301
